<a href="https://colab.research.google.com/github/anisaratna/ecommerce-analytics/blob/main/A_B_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
# 1. Login ke Google Cloud
auth.authenticate_user()
project_id = 'olist-ecommerce-493504'
client = bigquery.Client(project=project_id)

In [3]:
# 2. Ambil data nilai order
query = f"""
SELECT price, freight_value, total_order_value
FROM `{project_id}.raw_olist.master_order_data`
"""
df = client.query(query).to_dataframe()

In [8]:
# 3. SIMULASI
# Membagi pelanggan menjadi 2 kelompok secara acak
# Kelompok A: Free Shipping (Ongkir = 0)
# Kelompok B: Discount 10% (Harga barang dikurangi 10%)

np.random.seed(42)
df['group'] = np.random.choice(['Free_Shipping', 'Discount_10'], size=len(df))

def calculate_simulated_total(row):
    if row['group'] == 'Free_Shipping':
        return row['price']
    else:
        return (row['price'] * 0.9) + row['freight_value']

df['simulated_total'] = df.apply(calculate_simulated_total, axis=1)

In [9]:
print(df.head())

   price  freight_value  total_order_value          group  simulated_total
0   0.85          18.23              19.08  Free_Shipping            0.850
1   0.85          18.23              19.08    Discount_10           18.995
2   0.85          22.30              23.15  Free_Shipping            0.850
3   1.20           7.89               9.09  Free_Shipping            1.200
4   1.20           7.89               9.09  Free_Shipping            1.200


In [10]:
# Hitung rata-rata tiap kelompok
group_a = df[df['group'] == 'Free_Shipping']['simulated_total']
group_b = df[df['group'] == 'Discount_10']['simulated_total']

print(f"Rata-rata Kelompok A (Free Shipping): {group_a.mean():.2f}")
print(f"Rata-rata Kelompok B (Discount 10%): {group_b.mean():.2f}")

Rata-rata Kelompok A (Free Shipping): 119.07
Rata-rata Kelompok B (Discount 10%): 128.77


In [11]:
# Jalankan T-Test
t_stat, p_value = stats.ttest_ind(group_a, group_b)
print(f"\nT-statistic: {t_stat}")
print(f"P-value: {p_value}")

if p_value < 0.05:
    print("\nKESIMPULAN: Perbedaannya SIGNIFIKAN secara statistik. Satu strategi lebih unggul.")
else:
    print("\nKESIMPULAN: Perbedaannya TIDAK SIGNIFIKAN. Hasilnya hampir sama saja.")


T-statistic: -9.121446715182815
P-value: 7.532910590700601e-20

KESIMPULAN: Perbedaannya SIGNIFIKAN secara statistik. Satu strategi lebih unggul.
